In [16]:
import pandas as pd

df_training = pd.read_csv('data_spliting4/traing.csv')
df_test = pd.read_csv('data_spliting4/testing.csv')
df_movies = pd.read_csv('ml-latest-small/movies.csv')

In [17]:
df_user1 = df_training.loc[df_training['userId'] == 1]

df_result = df_user1.merge(df_movies, on='movieId', how='left')

generos = df_movies['genres'].str.split('|').explode().unique().tolist()



In [18]:
dummies = df_result['genres'].str.get_dummies(sep='|')
df_result = pd.concat([df_result, dummies], axis=1)
df_result
# df_result.to_csv('lista_generos_traing.csv', index=False)

,userId,movieId,rating,timestamp,titulo_movie_lens,genres,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
3,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,1,110,4.0,964982176,Braveheart (1995),Action|Drama|War,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,1,3740,4.0,964982417,Big Trouble in Little China (1986),Action|Adventure|Comedy|Fantasy,1,1,0,0,...,1,0,0,0,0,0,0,0,0,0
156,1,3793,5.0,964981855,X-Men (2000),Action|Adventure|Sci-Fi,1,1,0,0,...,0,0,0,0,0,0,1,0,0,0
157,1,3809,4.0,964981220,What About Bob? (1991),Comedy,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
158,1,4006,4.0,964982903,Transformers: The Movie (1986),Adventure|Animation|Children|Sci-Fi,0,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [19]:
df_binary = df_result.iloc[:, 6:]


soma =df_binary.sum().sort_values()
# soma.to_csv('validation/Treinamentosoma_generos_treinos.csv')

In [20]:
df_user1 = df_test.loc[df_test['userId'] == 1]

df_result = df_user1.merge(df_movies, on='movieId', how='left')

generos = df_movies['genres'].str.split('|').explode().unique().tolist()


In [21]:
dummies = df_result['genres'].str.get_dummies(sep='|')
df_result = pd.concat([df_result, dummies], axis=1)

# df_result.to_csv('validation/test/lista_generos_test.csv', index=False)

In [22]:
df_binary = df_result.iloc[:, 6:]

# soma =df_binary.sum().sort_values()
# soma.to_csv('validation/test/soma_generos_test.csv')

# Funções reutilizáveis para geração de df de gêneros e comparação quantitativa
import numpy as np

def compute_user_genre_counts(df, movies, user_id):
    df_user = df.loc[df['userId'] == user_id]
    if df_user.empty:
        return pd.DataFrame(), pd.Series(dtype='int')
    merged = df_user.merge(movies, on='movieId', how='left')
    dummies = merged['genres'].str.get_dummies(sep='|')
    merged = pd.concat([merged, dummies], axis=1)
    genre_counts = dummies.sum().sort_values(ascending=False)
    return merged, genre_counts


def compare_genre_profiles(user_id, train_df, test_df, movies):
    _, train_counts = compute_user_genre_counts(train_df, movies, user_id)
    _, test_counts = compute_user_genre_counts(test_df, movies, user_id)

    all_genres = sorted(set(train_counts.index) | set(test_counts.index))
    train_vec = train_counts.reindex(all_genres, fill_value=0).astype(float)
    test_vec = test_counts.reindex(all_genres, fill_value=0).astype(float)

    diff = (train_vec - test_vec).abs()
    euclidean = np.linalg.norm(train_vec - test_vec)
    dot = float(np.dot(train_vec, test_vec))
    norm_train = np.linalg.norm(train_vec)
    norm_test = np.linalg.norm(test_vec)
    cosine = dot / (norm_train * norm_test) if norm_train > 0 and norm_test > 0 else 0.0

    top_train = train_vec.idxmax() if norm_train > 0 else None
    top_test = test_vec.idxmax() if norm_test > 0 else None
    top_match = (top_train == top_test)

    summary = pd.DataFrame({'train': train_vec, 'test': test_vec, 'abs_diff': diff})
    summary = summary.sort_values('abs_diff', ascending=False)

    return {
        'user_id': user_id,
        'top_train': top_train,
        'top_test': top_test,
        'top_match': top_match,
        'total_diff': int(diff.sum()),
        'euclidean_distance': float(euclidean),
        'cosine_similarity': float(cosine),
        'genre_pairwise': summary,
    }


# Aplicar aos 250 primeiros usuários
results = []
for user_id in range(1, 601):
    results.append(compare_genre_profiles(user_id, df_training, df_test, df_movies))

summary_df = pd.DataFrame([{
    'userId': r['user_id'],
    'top_train': r['top_train'],
    'top_test': r['top_test'],
    'top_match': r['top_match'],
    'total_diff': r['total_diff'],
    'euclidean_distance': r['euclidean_distance'],
    'cosine_similarity': r['cosine_similarity']
} for r in results])

summary_df.to_csv('validation/user_genre_profile_comparison_2.csv', index=False)

print('Resumo 601 usuários:')
print(summary_df.head(10))
print('\nQuantidade de usuários com top genre diferente:', (~summary_df['top_match']).sum())
print('Porcentagem:', 100 * (~summary_df['top_match']).mean())

# Exemplo de detalhamento para user 1
print('\nDetalhe user 1:')
print(results[0]['genre_pairwise'].head(10))

Resumo 601 usuários:
   userId top_train   top_test  top_match  total_diff  euclidean_distance  \
0       1    Action     Comedy      False         352          103.169763   
1       2     Drama      Crime      False          19            8.062258   
2       3    Sci-Fi     Sci-Fi       True          22           10.198039   
3       4     Drama      Drama       True         170           63.292970   
4       5     Drama      Drama       True          36           11.313708   
5       6     Drama      Drama       True         209           66.037868   
6       7    Action     Comedy      False         123           43.139309   
7       8     Drama  Adventure      False          40           13.038405   
8       9     Drama     Comedy      False          34           11.747340   
9      10     Drama     Comedy      False          86           31.717503   

   cosine_similarity  
0           0.965255  
1           0.843274  
2           0.892725  
3           0.942483  
4           0.87

In [2]:
import pandas as pd

In [3]:
df_time_stamp = pd.read_csv('validation/user_genre_profile_comparison_1_250.csv')

In [4]:
df_80_20 = pd.read_csv('validation/user_genre_profile_comparison_2.csv')

In [5]:
mean_time_stamp = df_time_stamp.loc[:, 'total_diff'].mean()
mean_df_80_20 = df_80_20.loc[:, 'total_diff'].mean()


f'no df com time_stamp o {mean_time_stamp} e {mean_df_80_20}'


'no df com time_stamp o 170.05833333333334 e 125.72166666666666'

In [6]:
euclidean_mean_time_stamp = df_time_stamp.loc[:, 'euclidean_distance'].mean()
euclidean_mean_df_80_20 = df_80_20.loc[:, 'euclidean_distance'].mean()


f'Euclidean no df com time_stamp o {euclidean_mean_time_stamp} e {euclidean_mean_df_80_20}'

'Euclidean no df com time_stamp o 56.43633316562944 e 41.98015567192435'

In [7]:
cossine_mean_time_stamp = df_time_stamp.loc[:, 'cosine_similarity'].mean()
cossine_mean_df_80_20 = df_80_20.loc[:, 'cosine_similarity'].mean()


f'Euclidean no df com time_stamp o {cossine_mean_time_stamp} e {cossine_mean_df_80_20}'

'Euclidean no df com time_stamp o 0.7384121435393861 e 0.8409097067412921'